# Luganda word list from 48 Hugging Face datasets

Reads the Luganda text columns of every dataset below and counts each word form, writing `luganda_words.tsv`:

| column | meaning |
|---|---|
| `word` | the word, lowercased |
| `count` | occurrences in human-written text |
| `datasets` | how many human-written datasets have it |
| `synthetic_count` | occurrences in machine-translated or generated text, counted apart |
| `capitalised` | share of its mid-sentence occurrences written with a capital: near 1 for names |

Only sentences that read as Luganda are counted (ganda-grammar's own test), and within a dataset each sentence counts
once. Only the listed text columns are downloaded, never the audio or images stored beside them.

On Kaggle: Settings → Internet on. A dataset that fails is reported and skipped; each finished dataset is saved
to `parts/`, so a rerun resumes where the last one stopped.


In [ ]:
import collections, json, os, pickle, re, time, traceback, urllib.parse, urllib.request
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

OUT = Path("/kaggle/working" if Path("/kaggle").exists() else ".") / "luganda_words"
PARTS = OUT / "parts"
PARTS.mkdir(parents=True, exist_ok=True)
MIN_COUNT = 2   # a word written fewer times than this (human + synthetic) is left out of the list
WORKERS = os.cpu_count() or 2

# (repo, config, Luganda text columns, synthetic). A column path: "a", "a.b" (a field), "a[]" (a list of strings, joined),
# "a[].b" (field b of each item of list a).
DATASETS = [
    # human-written
    ("henryhale/luganda-pretrain2", "default", ["text"], False),
    ("goldfish-models/fish-food", "lug_latn", ["text"], False),
    ("sagepond/gec", "default", ["tgt"], False),  # tgt only: src holds deliberately misspelled text
    ("sagepond/luganda", "default", ["luganda"], False),
    ("sagepond/luganda", "paragraphs", ["luganda"], False),
    ("michsethowusu/ganda-sentiments-corpus", "default", ["Ganda"], False),
    ("cis-lmu/Glot500", "lug_Latn", ["text"], False),
    ("michsethowusu/english-ganda_sentence-pairs_mt560", "default", ["lug"], False),
    ("google/WaxalNLP", "lug_asr", ["transcription"], False),
    ("henryhale/luganda-pretrain", "default", ["text"], False),
    ("pkyoyetera/luganda_english_dataset", "default", ["Luganda"], False),
    ("henryhale/english-luganda-translation", "default", ["luganda"], False),
    ("sulaimank/luganda_text_wiki_bukedde_radio_flores", "default", ["text"], False),
    ("aceuganda/lacuna-luganda-health-corpus", "default", ["approved_translation"], False),
    ("HuggingFaceFW/fineweb-2", "lug_Latn", ["text"], False),
    ("HuggingFaceFW/finetranslations", "lug_Latn", ["og_full_text"], False),
    ("cis-lmu/Taxi1500-RawData", "lug_Latn", ["text"], False),
    ("multilingual-tts/open-bible", "Luganda", ["text"], False),
    ("cyanic-selkie/wikianc", "lg", ["paragraph_text"], False),
    ("HPLT/HPLT2.0_cleaned", "lug_Latn", ["text"], False),
    ("AfriSpeech/madlad-91-african", "lg_clean", ["local"], False),
    ("kukuzaai/lug-cleaned-200M-tokens", "default", ["headline", "full_body"], False),
    ("KMayanja/merged_txt_eng_lug", "default", ["Luganda"], False),
    ("keystats/luganda_asr_dataset", "default", ["transcription"], False),
    ("akoksal/muri-it-language-split", "lug", ["output"], False),
    ("sbmaruf/forai_ml_masakhane_mafand", "en-lug", ["target"], False),
    ("masakhane/masakhaner2", "lug", ["tokens[]"], False),
    ("wikimedia/wikidata-title-desc", "lg", ["description"], False),
    ("allandclive/UgandaLex", "default", ["Ganda"], False),
    ("allandclive/UgandaLex2", "default", ["Ganda"], False),
    ("gimmy256/ugandan_languages", "default", ["Luganda Translation"], False),
    ("alexandrainst/multi-wiki-qa", "lg", ["context"], False),
    # machine-translated or generated
    ("henryhale/tinystories-ganda", "default", ["tgt"], True),
    ("bigscience/xP3all", "lg", ["targets"], True),
    ("michsethowusu/Code-170k-luganda", "default", ["conversations[].value"], True),
    ("yigagilbert/synthetic-parallel-external", "default", ["text_lug"], True),
    ("yigagilbert/synthetic-parallel-sunbird", "default", ["text_lug"], True),
    ("yigagilbert/synthetic-parallel-salt", "default", ["text_lug"], True),
    ("henryhale/luganda-sft-translated", "default", ["instruction_lug", "output_lug"], True),
    ("henryhale/luganda-simplestories", "default", ["text"], True),
    ("saillab/alpaca_luganda_taco", "default", ["instruction", "input", "output"], True),
    ("saillab/alpaca-luganda-cleaned", "default", ["instruction", "input", "output"], True),
    ("henryhale/luganda-ultrafeedback-dpo", "default", ["prompt", "chosen[].content", "rejected[].content"], True),
    ("henryhale/luganda-smol-smoltalk-40k", "default", ["conversations[].content"], True),
    ("henryhale/luganda-reasoning", "default", ["question", "reasoning"], True),
    ("anyantudre/waxal-linsna", "pseudo_agree_lug", ["text"], True),  # transcripts made by a speech model
    ("Tadesse/COMET_score", "eng_lug", ["Luganda"], True),
    ("gimmy256/afri-aya-vision", "default", ["question_local", "answer_local", "caption_local"], True),
]
# henryhale/luganda-pretrain2 also holds tinystories-ganda (machine-translated), already counted as synthetic above
SKIP_ROWS = {"henryhale/luganda-pretrain2": lambda row: row.get("source") == "tinystories-ganda"}
EXTRA_COLUMNS = {"henryhale/luganda-pretrain2": ["source"]}
assert len({(r, c) for r, c, *_ in DATASETS}) == len(DATASETS), "a repo and config listed twice"
print(len(DATASETS), "datasets:", sum(not d[3] for d in DATASETS), "human-written,", sum(d[3] for d in DATASETS), "synthetic")


In [ ]:
# Reading: the Hugging Face dataset viewer lists every dataset as parquet files; only the needed columns are fetched
import pyarrow.parquet as pq
from fsspec.parquet import open_parquet_file  # fetches only the byte ranges of the columns asked for

from ganda_grammar.rules.base import looks_luganda, luganda_tokens, sentences

API = "https://datasets-server.huggingface.co"


def get_json(path, **params):
    url = f"{API}/{path}?" + urllib.parse.urlencode(params)
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return json.load(r)
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * 2 ** attempt)


def parquet_files(repo, config):
    files = get_json("parquet", dataset=repo, config=config)["parquet_files"]
    assert files, "no parquet files"
    return [f["url"] for f in files]


def rows(url, columns):
    """The rows of one parquet file with only these top-level columns (audio and images beside them aren't fetched)."""
    for attempt in range(5):
        try:
            with open_parquet_file(url, columns=columns, engine="pyarrow") as f:
                pf = pq.ParquetFile(f)
                names = set(pf.schema_arrow.names)
                missing = [c for c in columns if c not in names]
                assert not missing, f"no column {missing} in {sorted(names)[:20]}"
                batches = list(pf.iter_batches(batch_size=10_000, columns=columns))
            break
        except AssertionError:
            raise
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * 2 ** attempt)
    for b in batches:
        yield from b.to_pylist()


def texts(value, path):
    """The strings at a column path: text, a.b, a[], a[].b."""
    if not path:
        if isinstance(value, str):
            yield value
        return
    head, _, rest = path.partition(".")
    is_list = head.endswith("[]")
    if head.rstrip("[]"):
        value = value.get(head.rstrip("[]")) if isinstance(value, dict) else None
    if value is None:
        return
    if is_list and not rest:  # a list of strings is one text: the words of a sentence (tokens[])
        yield " ".join(x for x in value if isinstance(x, str)) if isinstance(value, list) else ""
    elif is_list:
        for item in value if isinstance(value, list) else []:
            yield from texts(item, rest)
    else:
        yield from texts(value, rest)


In [ ]:
# Counting one dataset (run in a worker process; the result is saved to parts/)
def part_file(repo, config):
    return PARTS / (re.sub(r"[^\w.-]", "_", f"{repo}__{config}") + ".pkl")


def count_dataset(repo, config, columns, synthetic):
    start = time.time()
    counts, capitals = collections.Counter(), collections.Counter()
    seen = set()  # each sentence counts once within the dataset
    stats = collections.Counter()
    tops = sorted({c.split(".")[0].rstrip("[]") for c in columns} | set(EXTRA_COLUMNS.get(repo, [])))
    skip = SKIP_ROWS.get(repo)
    for url in parquet_files(repo, config):
        for row in rows(url, tops):
            stats["rows"] += 1
            if skip and skip(row):
                stats["rows skipped"] += 1
                continue
            for col in columns:
                for text in texts(row, col):
                    for s, e in sentences(text):
                        stats["sentences"] += 1
                        key = hash(" ".join(text[s:e].lower().split()))
                        if key in seen:
                            continue
                        seen.add(key)
                        toks = luganda_tokens(text, s, e)
                        if not toks:
                            continue
                        stats["luganda sentences"] += 1
                        for i, t in enumerate(toks):
                            w = t.text.lower()
                            if looks_luganda(w):
                                counts[w] += 1
                                if i and t.text[0].isupper():
                                    capitals[w] += 1
    stats["words"] = sum(counts.values())
    stats["seconds"] = round(time.time() - start)
    result = dict(repo=repo, config=config, synthetic=synthetic, counts=counts, capitals=capitals, stats=dict(stats))
    with open(part_file(repo, config), "wb") as f:
        pickle.dump(result, f)
    return repo, config, stats


def run(d):
    try:
        return count_dataset(*d)
    except Exception as e:
        return d[0], d[1], {"error": f"{type(e).__name__}: {str(e)[:300]}"}


In [ ]:
# Count every dataset not done yet, several at a time
todo = [d for d in DATASETS if not part_file(d[0], d[1]).exists()]
print(f"{len(DATASETS) - len(todo)} done before, {len(todo)} to count with {WORKERS} workers")
errors = {}
with ProcessPoolExecutor(WORKERS, mp_context=multiprocessing.get_context("fork")) as pool:
    jobs = [pool.submit(run, d) for d in todo]
    for job in as_completed(jobs):
        repo, config, stats = job.result()
        if "error" in stats:
            errors[f"{repo} [{config}]"] = stats["error"]
            print(f"FAILED {repo} [{config}]: {stats['error']}")
        else:
            print(f"{repo} [{config}]: {stats.get('rows', 0):,} rows, {stats.get('luganda sentences', 0):,} Luganda "
                  f"sentences, {stats.get('words', 0):,} words, {stats['seconds']}s")
print(f"{len(errors)} failed" + (": rerun this cell to retry them" if errors else ""))


In [ ]:
# Merge the datasets into one list
count, synthetic, capitals = collections.Counter(), collections.Counter(), collections.Counter()
datasets, synthetic_datasets = collections.Counter(), collections.Counter()
report = []
for repo, config, _, syn in DATASETS:
    p = part_file(repo, config)
    if not p.exists():
        report.append(dict(repo=repo, config=config, synthetic=syn, status="failed: " + errors.get(f"{repo} [{config}]", "not counted")))
        continue
    with open(p, "rb") as f:
        r = pickle.load(f)
    (synthetic if syn else count).update(r["counts"])
    (synthetic_datasets if syn else datasets).update(r["counts"].keys())
    capitals.update(r["capitals"])
    report.append(dict(repo=repo, config=config, synthetic=syn, status="ok", **r["stats"]))

words = sorted(set(count) | set(synthetic), key=lambda w: (-count[w], -synthetic[w], w))
kept = 0
with open(OUT / "luganda_words.tsv", "w", encoding="utf-8") as f:
    f.write("word\tcount\tdatasets\tsynthetic_count\tsynthetic_datasets\tcapitalised\n")
    for w in words:
        n = count[w] + synthetic[w]
        if n < MIN_COUNT:
            continue
        kept += 1
        f.write(f"{w}\t{count[w]}\t{datasets[w]}\t{synthetic[w]}\t{synthetic_datasets[w]}\t{capitals[w] / n:.2f}\n")

cols = ["repo", "config", "synthetic", "status", "rows", "rows skipped", "sentences", "luganda sentences", "words", "seconds"]
with open(OUT / "datasets.tsv", "w", encoding="utf-8") as f:
    f.write("\t".join(cols) + "\n")
    for r in report:
        f.write("\t".join(str(r.get(c, "")) for c in cols) + "\n")

human = sum(r.get("words", 0) for r in report if not r["synthetic"])
machine = sum(r.get("words", 0) for r in report if r["synthetic"])
print(f"{sum(r['status'] == 'ok' for r in report)} of {len(report)} datasets counted")
print(f"{human:,} words of human-written text, {machine:,} synthetic")
print(f"{len(words):,} distinct words, {kept:,} seen at least {MIN_COUNT} times -> {OUT / 'luganda_words.tsv'}")


In [ ]:
# A look: the commonest words, and words seen in many datasets but rarely
for w in words[:30]:
    print(f"{w:20} {count[w]:>12,} {datasets[w]:>3} datasets")
print()
spread = [w for w in words if datasets[w] >= 10 and count[w] < 50][:30]
print("in 10+ datasets but under 50 times:", ", ".join(spread))
